In [ ]:
!pip install -q -U transformers accelerate timm torch datasets
!pip install -q sacremoses sentencepiece # mBART dependencies

# --- Core Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, UnidentifiedImageError
from transformers import (
    MBart50TokenizerFast,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)
from transformers.modeling_outputs import BaseModelOutput
import timm
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import os
import json
import pandas as pd
import random
from transformers.modeling_outputs import BaseModelOutput
from contextlib import nullcontext
import unicodedata

In [ ]:
# --- Device Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Prefer TF32 on Ampere+; harmless elsewhere
import torch
torch.backends.cuda.matmul.allow_tf32 = True                
torch.backends.cudnn.allow_tf32 = True  

DATA_DIR = "/kaggle/input/synthetic-bic-pairs/synthetic_bengali_images"
# DATA_DIR = "/kaggle/input/syn-images/syn-ben-images"
REAL_IMAGE_BASE_DIR = "/kaggle/input/coco-image-caption/train2014/train2014"
COCO_ANNOTATIONS_PATH = '/kaggle/input/coco-image-caption/annotations_trainval2014/annotations/captions_train2014.json'
MODEL_SAVE_PATH = "/kaggle/working/maxvit_mbart_captioning_model.pth"
MODEL_INPUT_PATH = "/kaggle/input/train/transformers/default/12/maxvit_mbart_captioning_model.pth"
MODEL_CHECKPOINT_PATH = "/kaggle/working/maxvit_mbart_bn_checkpoint.pt"
MODEL_CHECKPOINT_PATH_IN = "/kaggle/input/train/transformers/default/15/maxvit_mbart_bn_checkpoint.pt"

# --- Model and Tokenizer Names ---
MAXVIT_MODEL_NAME = "maxvit_base_tf_224.in1k"
TOKENIZER_MODEL_NAME = "google/mt5-base"
DECODER_MODEL_NAME = "google/mt5-base"
MBART_MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt" 

# --- Training Configuration (Adjust as needed) ---
BATCH_SIZE = 2                                               
GRAD_ACCUM_STEPS = 4                                        
LEARNING_RATE = 1e-4                                       
NUM_EPOCHS = 7
PATCH_ALIGNMENT_LOSS_WEIGHT = 0.5
MAX_CAPTION_LENGTH = 96                                   
# INITIAL_PATCH_ALIGNMENT_WEIGHT = 0.05
INITIAL_PATCH_ALIGNMENT_WEIGHT = 0.0

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    transforms.ToTensor(),
    # Use ImageNet mean and std for normalization
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# --- Utility Functions for Data Loading ---

def extract_captions(full_caption):
    if "In Bengali:" in full_caption:
        caption_en = full_caption.split("In Bengali:")[0].strip()
        caption_bn = full_caption.split("In Bengali:")[-1].strip()
    else:
        parts = full_caption.strip().split(". ")
        if len(parts) >= 2:
            caption_en = parts[0].strip()
            caption_bn = parts[-1].strip()
        else:
            caption_en = full_caption.strip()
            caption_bn = full_caption.strip()
    if caption_en.startswith("A photo of: "):
        caption_en = caption_en[len("A photo of: "):].strip()
    return caption_en, caption_bn

def handle_full_stop_variation(caption_en):
    captions = [caption_en]
    if caption_en.endswith('.'):
        captions.append(caption_en[:-1].strip())
    if not caption_en.endswith('.'):
        captions.append(caption_en + '.')
    return captions

def safe_load_image(img_path: str):
    try:
        img = Image.open(img_path).convert('RGB')
        return img
    except FileNotFoundError:
        print(f"ERROR: Image file not found at: {img_path}")
        return None
    except UnidentifiedImageError:
        print(f"ERROR: Cannot identify image file (corrupted or unsupported format): {img_path}")
        return None
    except Exception as e:
        print(f"ERROR: An unexpected error occurred while loading image {img_path}: {e}")
 
        return None


def clean_bengali_caption(caption_text):
    cleaned = unicodedata.normalize('NFKC', str(caption_text))
    cleaned = (cleaned.replace('‘', "'").replace('’', "'")
                     .replace('“', '"').replace('”', '"')
                     .replace('—', '-').replace('–', '-')
                     .replace('…', '...'))
    cleaned = ''.join(ch for ch in cleaned if ch.isprintable())
    cleaned = ' '.join(cleaned.split()).strip()
    # strip any lone surrogates / odd bytes that can upset the fast tokenizer
    cleaned = cleaned.encode('utf-8', 'ignore').decode('utf-8', 'ignore')   # <<< added
    return cleaned

# --- Custom Dataset Class (Using torchvision transforms) ---
class BengaliCaptionDataset(Dataset):
    def __init__(self, df_results, image_transform, tokenizer, max_length=MAX_CAPTION_LENGTH):
        self.df = df_results
        self.image_transform = image_transform
        self.tokenizer = tokenizer
        self.max_length = int(max_length)
        self.pad_id = int(self.tokenizer.pad_token_id)  
        
        self.tokenizer.src_lang = "bn_IN"
        self.tokenizer.tgt_lang = "bn_IN"

        print(f"Dataset initialized with {len(self.df)} entries.")
        
        self.successfully_tokenized_count = 0
        self.failed_tokenization_count = 0



    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        real_image_path = row['real_image_path']
        generated_image_path = row['generated_image_path']
        caption_bn = row["caption_bn"]
        caption_bn = clean_bengali_caption(caption_bn)

        real_img = safe_load_image(real_image_path)
        generated_img = safe_load_image(generated_image_path)

        if real_img is None or generated_img is None:
            return None

        try:

            if not isinstance(real_img, Image.Image):
                return None # Ensure it's skipped
            if not isinstance(generated_img, Image.Image):
                return None # Ensure it's skipped
            
            real_pixel_values = self.image_transform(real_img)
           
            synthetic_pixel_values = self.image_transform(generated_img)
           
        except Exception as e:
            return None

      
        try:
        
            try:
                tokens = self.tokenizer(
                    caption_bn,
                    return_tensors="pt",
                    truncation=True,
                    max_length=int(self.max_length),             
                    padding="max_length"
                )
            except OverflowError:
                if not hasattr(self, "_tok_slow"):
                    self._tok_slow = MBart50TokenizerFast.from_pretrained(
                        "facebook/mbart-large-50-many-to-many-mmt",
                        src_lang="bn_IN", tgt_lang="bn_IN"
                    )
                tokens = self._tok_slow(
                    caption_bn,
                    return_tensors="pt",
                    truncation=True,
                    max_length=int(self.max_length),
                    padding="max_length"
                )
            except Exception as e:
                print(f"Tokenizer failed for idx {idx}: {e}")
                return None

            labels = tokens.input_ids.squeeze(0)
            attention_mask = tokens["attention_mask"].squeeze(0)
            
            # mask PAD tokens in labels so CE ignores them
            pad_id = self.tokenizer.pad_token_id
            labels[labels == pad_id] = -100
            
            # guard: if everything is ignored, skip sample to avoid CE NaN
            if torch.all(labels == -100):
                return None
            
            self.successfully_tokenized_count += 1

        except Exception as e:
            print(f"CRITICAL ERROR during tokenization for index {idx}")
            print(f"  Problematic caption: '{caption_bn}'")
            print(f"  Type of problematic caption: {type(caption_bn)}")
            print(f"  Length of problematic caption: {len(caption_bn)}")
            print(f"  FULL EXCEPTION: {type(e).__name__}: {e}")
            self.failed_tokenization_count += 1
            return None

        return {
            "real_pixel_values": real_pixel_values,
            "synthetic_pixel_values": synthetic_pixel_values,
            "labels": labels,
            "attention_mask": attention_mask
        }

# --- Custom Collate Function for DataLoader ---
def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch:
        return None

    real_pixel_values = torch.stack([item["real_pixel_values"] for item in batch])
    synthetic_pixel_values = torch.stack([item["synthetic_pixel_values"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])

    return {
        "real_pixel_values": real_pixel_values,
        "synthetic_pixel_values": synthetic_pixel_values,
        "labels": labels,
        "attention_mask": attention_mask
    }



In [ ]:
from collections import Counter
import traceback
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # cope with truncated JPEGs
from transformers.modeling_outputs import BaseModelOutput  # if not already imported


# --- Debug wrapper to pinpoint failing transform op ---
class DebugCompose(torch.nn.Module):
    def __init__(self, ops):
        super().__init__()
        self.ops = ops
    def forward(self, img, path_hint=""):
        x = img
        for i, op in enumerate(self.ops):
            try:
                x = op(x)
            except Exception as e:
                print(f"\n[TRANSFORM FAIL] file={path_hint}\n  op#{i}: {op}\n  error: {type(e).__name__}: {e}")
                traceback.print_exc(limit=1)
                raise
        return x

# --- Safer RandomResizedCrop with fallback so the sample is never dropped ---
class SafeRandomResizedCrop(torch.nn.Module):
    def __init__(self, size, scale=(0.5, 1.0), ratio=(0.75, 1.3333)):
        super().__init__()
        self.rrc = transforms.RandomResizedCrop(size, scale=scale, ratio=ratio)
        self.fallback = transforms.Compose([transforms.Resize((size, size)), transforms.CenterCrop(size)])
    def forward(self, img):
        try:
            return self.rrc(img)
        except Exception:
            return self.fallback(img)

# --- (A) DEBUG transform (only for preflight) ---
debug_ops = [
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),  # original – we want to see if this fails
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]
debug_transform = DebugCompose(debug_ops)

# --- (B) SAFE training transform (use for actual training) ---
train_transform_safe = transforms.Compose([
    transforms.Resize((256, 256)),
    SafeRandomResizedCrop(224, scale=(0.5, 1.0)),  # more forgiving + fallback
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.0),  # disable; can hurt caption semantics
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



def preflight_filter(df_results, tokenizer, image_transform, max_length, use_debug=False):
    """
    Validates each row once, logs reasons, and returns indices to keep.
    If use_debug=True, prints the exact op/file that failed.
    """
    reasons = Counter()
    keep = []

    for i, row in enumerate(tqdm(df_results.itertuples(index=True), total=len(df_results), desc="Preflight")):
        # 1) load images
        real_img = safe_load_image(row.real_image_path)
        if real_img is None:
            reasons["real_image_load_fail"] += 1; continue
        gen_img  = safe_load_image(row.generated_image_path)
        if gen_img is None:
            reasons["gen_image_load_fail"] += 1; continue

        # 2) transforms
        try:
            if use_debug:
                _ = image_transform(real_img, row.real_image_path)
                _ = image_transform(gen_img,  row.generated_image_path)
            else:
                _ = image_transform(real_img)
                _ = image_transform(gen_img)
        except Exception:
            reasons["transform_fail"] += 1; continue

        # 3) tokenization
        text = clean_bengali_caption(row.caption_bn)
        try:
            toks = tokenizer(text, return_tensors="pt", truncation=True,
                             max_length=int(max_length), padding="max_length")
        except Exception:
            reasons["tokenize_fail"] += 1; continue

        input_ids = toks["input_ids"].squeeze(0)
        attn_mask = toks["attention_mask"].squeeze(0)
        pad_id = int(tokenizer.pad_token_id)

        labels = input_ids.clone()
        labels[labels == pad_id] = -100

        if attn_mask.sum().item() == 0:
            reasons["all_pad_attn_mask"] += 1; continue
        if torch.all(labels == -100):
            reasons["all_ignore_labels"] += 1; continue

        keep.append(row.Index)

    print("\nPreflight summary:", dict(reasons))
    return keep

In [ ]:
from collections import Counter
import traceback
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # cope with truncated JPEGs
from transformers.modeling_outputs import BaseModelOutput  # if not already imported


# --- Debug wrapper to pinpoint failing transform op ---
class DebugCompose(torch.nn.Module):
    def __init__(self, ops):
        super().__init__()
        self.ops = ops
    def forward(self, img, path_hint=""):
        x = img
        for i, op in enumerate(self.ops):
            try:
                x = op(x)
            except Exception as e:
                print(f"\n[TRANSFORM FAIL] file={path_hint}\n  op#{i}: {op}\n  error: {type(e).__name__}: {e}")
                traceback.print_exc(limit=1)
                raise
        return x

# --- Safer RandomResizedCrop with fallback so the sample is never dropped ---
class SafeRandomResizedCrop(torch.nn.Module):
    def __init__(self, size, scale=(0.5, 1.0), ratio=(0.75, 1.3333)):
        super().__init__()
        self.rrc = transforms.RandomResizedCrop(size, scale=scale, ratio=ratio)
        self.fallback = transforms.Compose([transforms.Resize((size, size)), transforms.CenterCrop(size)])
    def forward(self, img):
        try:
            return self.rrc(img)
        except Exception:
            return self.fallback(img)

# --- (A) DEBUG transform ---
debug_ops = [
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]
debug_transform = DebugCompose(debug_ops)

# --- (B) SAFE training transform ---
train_transform_safe = transforms.Compose([
    transforms.Resize((256, 256)),
    SafeRandomResizedCrop(224, scale=(0.5, 1.0)),  # more forgiving + fallback
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.0),  # disable; can hurt caption semantics
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



def preflight_filter(df_results, tokenizer, image_transform, max_length, use_debug=False):
    """
    Validates each row once, logs reasons, and returns indices to keep.
    If use_debug=True, prints the exact op/file that failed.
    """
    reasons = Counter()
    keep = []

    for i, row in enumerate(tqdm(df_results.itertuples(index=True), total=len(df_results), desc="Preflight")):
        # 1) load images
        real_img = safe_load_image(row.real_image_path)
        if real_img is None:
            reasons["real_image_load_fail"] += 1; continue
        gen_img  = safe_load_image(row.generated_image_path)
        if gen_img is None:
            reasons["gen_image_load_fail"] += 1; continue

        # 2) transforms
        try:
            if use_debug:
                _ = image_transform(real_img, row.real_image_path)
                _ = image_transform(gen_img,  row.generated_image_path)
            else:
                _ = image_transform(real_img)
                _ = image_transform(gen_img)
        except Exception:
            reasons["transform_fail"] += 1; continue

        # 3) tokenization
        text = clean_bengali_caption(row.caption_bn)
        try:
            toks = tokenizer(text, return_tensors="pt", truncation=True,
                             max_length=int(max_length), padding="max_length")
        except Exception:
            reasons["tokenize_fail"] += 1; continue

        input_ids = toks["input_ids"].squeeze(0)
        attn_mask = toks["attention_mask"].squeeze(0)
        pad_id = int(tokenizer.pad_token_id)

        labels = input_ids.clone()
        labels[labels == pad_id] = -100

        if attn_mask.sum().item() == 0:
            reasons["all_pad_attn_mask"] += 1; continue
        if torch.all(labels == -100):
            reasons["all_ignore_labels"] += 1; continue

        keep.append(row.Index)

    print("\nPreflight summary:", dict(reasons))
    return keep

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.backends.backend_pdf import PdfPages

import os, glob, math
os.environ["HISTORY_CSV_PATH"] = "/kaggle/working/history_with_ce.csv"  
os.environ["COMPARE_WITH_CSV"]  = "/kaggle/input/history/history_with_pal.csv"
os.environ["LOSS_FIG_PATH"]     = "/kaggle/working/loss_pal_vs_ce.pdf"


# -----------------------------
# Output paths for logs/figures
# -----------------------------

RUN_LABEL = os.getenv("RUN_LABEL", "with_pal")
RUN_ID    = os.getenv("RUN_ID", None)


HISTORY_CSV_PATH = os.getenv("HISTORY_CSV_PATH", "/kaggle/working/history_current_run.csv")


COMPARE_WITH_CSV = os.getenv("COMPARE_WITH_CSV", "")


LOSS_FIG_PATH = os.getenv("LOSS_FIG_PATH", "/kaggle/working/loss_pal_vs_ce.pdf")

# --- Needed imports for inset plots ---
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset


def _env_bool(name: str, default: str = "0") -> bool:
    v = os.getenv(name, default).strip().lower()
    return v in ("1","true","yes","y","on")

LOGY_TOTAL = _env_bool("LOGY_TOTAL", "0")

def _env_inset(default="1,3"):
    raw = os.getenv("INSET_EPOCHS", default)
    try:
        a,b = raw.split(",")
        return (int(a), int(b))
    except Exception:
        return (1,3)
INSET_EPOCHS = _env_inset()


def _drop_unnamed(df: pd.DataFrame) -> pd.DataFrame:
    """Drop stray index columns like 'Unnamed: 0'."""
    return df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]

def _coerce_numeric(df: pd.DataFrame, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def _guess_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """Return first existing column among candidates (case-insensitive, substring ok)."""
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    for c in cols:
        cl = c.lower()
        if any(cand.lower() in cl for cand in candidates):
            return c
    return None

def _ensure_run_id(df: pd.DataFrame, fallback_label: str) -> pd.DataFrame:
    """
    Ensure a column 'run_id' exists. Try 'run_id','seed','trial','fold';
    otherwise assign a single run id derived from fallback_label.
    """
    for k in ["run_id","seed","trial","fold"]:
        if k in df.columns:
            df["run_id"] = df[k].astype(str)
            break
    else:
        df["run_id"] = fallback_label  # single-run file
    return df

def _per_epoch_mean_ci(df: pd.DataFrame, key: str) -> pd.DataFrame:
    """
    Compute per-epoch mean and 95% CI across runs.
    If multiple rows per (epoch, run_id), we first average within run, then across runs.
    """
    if key not in df.columns:
        # produce empty frame with required keys
        return pd.DataFrame({"epoch": [], f"{key}_mean": [], f"{key}_ci": []})

    # average within each run_id per epoch (handles multiple minibatch logs per epoch)
    g = df.groupby(["run_id","epoch"], as_index=False)[key].mean()
    # aggregate across runs for each epoch
    agg = g.groupby("epoch")[key].agg(["mean","std","count"]).reset_index()
    # 95% CI with normal approx: 1.96 * std / sqrt(n)
    ci = 1.96 * (agg["std"] / np.sqrt(agg["count"].clip(lower=1)))
    out = pd.DataFrame({
        "epoch": agg["epoch"].astype(int),
        f"{key}_mean": agg["mean"].astype(float),
        f"{key}_ci":   ci.fillna(0.0).astype(float),
    }).sort_values("epoch").reset_index(drop=True)
    return out


# ============================================================
# A) Lightweight training logger
# ============================================================
class TrainHistory:
    """
    Collects per-epoch metrics and can save/load as CSV.
    Usage:
        H = TrainHistory(run_label="with_pal", run_id="seed42")
        H.log_epoch(epoch, total=..., ce=..., pal=..., nce=..., ot=...)
        H.to_csv("history_with_pal.csv")
    """
    def __init__(self, run_label: str, run_id: str | None = None, meta: dict | None = None):
        self._rows = []
        self.run_label = run_label
        self.run_id = run_id
        self.meta = meta or {}

    def log_epoch(self, epoch: int, **metrics):
        row = {"epoch": int(epoch), "run_label": self.run_label}
        if self.run_id is not None:
            row["run_id"] = self.run_id
        row.update(metrics)
        self._rows.append(row)

    def to_df(self) -> pd.DataFrame:
        df = pd.DataFrame(self._rows)
        for c in ["total","ce","pal","nce","ot"]:
            if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
        return df.sort_values("epoch").reset_index(drop=True)

    def to_csv(self, path: str):
        df = self.to_df()
        df.to_csv(path, index=False)
        print(f"Saved training history to: {path}")

    @staticmethod
    def from_csv(path: str) -> pd.DataFrame:
        return pd.read_csv(path)

# ============================================================
# B) Helpers for plotting / aggregation
# ============================================================
def _ema(x: np.ndarray, alpha: float = 0.5):
    if len(x) == 0: return x
    y = np.zeros_like(x, dtype=float)
    y[0] = x[0]
    for i in range(1, len(x)):
        y[i] = alpha * x[i] + (1 - alpha) * y[i-1]
    return y


def _load_glob(glob_pat: str, label_alias: str):
    paths = sorted(glob.glob(glob_pat))
    if len(paths)==0:
        raise FileNotFoundError(f"No CSV matched: {glob_pat}")
    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        rid = os.path.splitext(os.path.basename(p))[0]
        dfs.append(_standardize_hist(df, label_alias, rid))
    return pd.concat(dfs, ignore_index=True)

def _per_epoch_mean_ci(df: pd.DataFrame, value_col: str):
    """Return mean and 95% CI across runs per epoch."""
    g = df.groupby(["epoch"], as_index=False)[value_col]
    tmp = g.agg(["mean","std","count"]).reset_index()
    tmp.columns = ["epoch","mean","std","count"]
    # 95% CI (Gaussian approx); if single run, CI=0
    ci = 1.96 * (tmp["std"] / tmp["count"].clip(lower=1).pow(0.5))
    out = pd.DataFrame({"epoch": tmp["epoch"], f"{value_col}_mean": tmp["mean"], f"{value_col}_ci": ci.fillna(0.0)})
    return out


import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42

def plot_pal_vs_ce(
    hist_with_pal: pd.DataFrame,
    hist_ce_only: pd.DataFrame,
    title: str = "Training Loss (PAL vs. CE-only)",
    use_log_scale: bool | None = None,
    annotate_delta: bool = True,
    figsize=(9, 5.5),
    dpi=140,
    out_path: str | None = None
):
    def _prep_df(df: pd.DataFrame, label: str) -> pd.DataFrame:
        df = df.copy()
        if "Epoch" in df.columns and "Total" in df.columns:
            df.rename(columns={"Epoch": "epoch", "Total": "total"}, inplace=True)
        if "epoch" not in df.columns or "total" not in df.columns:
            raise ValueError(f"{label}: expected columns 'epoch' and 'total' in history csv.")
        alpha = 0.35
        sm, cur = [], None
        for v in df["total"].tolist():
            cur = v if cur is None else alpha * v + (1 - alpha) * cur
            sm.append(cur)
        df["total_smooth"] = sm
        df["run_label"] = label
        return df

    A = _prep_df(hist_with_pal, "With PAL")
    B = _prep_df(hist_ce_only,  "CE-only")

    common_epochs = np.intersect1d(A["epoch"].to_numpy(), B["epoch"].to_numpy())
    A = A[A["epoch"].isin(common_epochs)].reset_index(drop=True)
    B = B[B["epoch"].isin(common_epochs)].reset_index(drop=True)
    if len(A) == 0 or len(B) == 0:
        raise ValueError("No overlapping epochs between the two runs.")

    if use_log_scale is None:
        final_gap_ratio = max(A["total"].iloc[-1], B["total"].iloc[-1]) / max(1e-8, min(A["total"].iloc[-1], B["total"].iloc[-1]))
        use_log_scale = (final_gap_ratio >= 3.0)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.grid(True, alpha=0.25, linestyle="--", linewidth=0.7)

    l1_raw,  = ax.plot(A["epoch"], A["total"], alpha=0.25, linewidth=1.0, label="With PAL (raw)")
    l2_raw,  = ax.plot(B["epoch"], B["total"], alpha=0.25, linewidth=1.0, label="CE-only (raw)")
    l1_s,    = ax.plot(A["epoch"], A["total_smooth"], linewidth=2.5, label="With PAL (smoothed)")
    l2_s,    = ax.plot(B["epoch"], B["total_smooth"], linewidth=2.5, linestyle="--", label="CE-only (smoothed)")

    y1 = A["total_smooth"].to_numpy()
    y2 = B["total_smooth"].to_numpy()
    ax.fill_between(A["epoch"], np.minimum(y1, y2), np.maximum(y1, y2), alpha=0.12)

    a_min_idx = int(np.argmin(y1))
    b_min_idx = int(np.argmin(y2))
    ax.scatter([A["epoch"].iloc[a_min_idx]], [y1[a_min_idx]], s=45, zorder=3)
    ax.scatter([B["epoch"].iloc[b_min_idx]], [y2[b_min_idx]], s=45, zorder=3)

    if annotate_delta:
        e = A["epoch"].iloc[-1]
        delta = B["total_smooth"].iloc[-1] - A["total_smooth"].iloc[-1]
        ax.annotate(
            f"Δ @ epoch {e}: {delta:.3f}",
            xy=(e, A["total_smooth"].iloc[-1]),
            xytext=(e, max(A["total_smooth"].max(), B["total_smooth"].max())*1.03),
            ha="right",
            arrowprops=dict(arrowstyle="->", lw=1.0)
        )

    if use_log_scale:
        ax.set_yscale("log")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training loss" + (" (log)" if use_log_scale else ""))
    ax.set_title(title, pad=12)
    ax.legend(handles=[l1_s, l2_s], loc="upper right", frameon=True)

    txt = "Loss curves showing the effect of Patch Alignment Loss (PAL) during training."
    bbox = FancyBboxPatch((0.01, -0.22), 0.98, 0.16, transform=ax.transAxes,
                          boxstyle="round,pad=0.4", linewidth=0.8, alpha=0.05)
    ax.add_patch(bbox)
    ax.text(0.02, -0.16, txt, transform=ax.transAxes, va="top")

    plt.tight_layout()
    pdf_path = out_path or "loss_pal_vs_ce.pdf"
    if not pdf_path.lower().endswith(".pdf"):
        pdf_path += ".pdf"
    with PdfPages(pdf_path) as pdf:
        pdf.savefig(fig, bbox_inches="tight", metadata={"Title": title})
    print(f"Saved comparison figure (PDF) to: {pdf_path}")
    plt.show()


In [ ]:
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

def _ensure_canvas(fig):
    """Attach an Agg canvas if missing and draw once to materialize a renderer."""
    if not hasattr(fig, "canvas") or fig.canvas is None or not hasattr(fig.canvas, "get_renderer"):
        FigureCanvas(fig)
    # Make sure renderer exists
    fig.canvas.draw()


# ---------- DEBUG SWITCH ----------
DEBUG_HISTORY = os.getenv("DEBUG_HISTORY", "1").strip().lower() in ("1","true","yes","y","on")

def _dbg(*args, **kwargs):
    if DEBUG_HISTORY:
        print(*args, **kwargs)

# ---------- Drop-in debug validators ----------
def _validate_df_for_hist(df: pd.DataFrame, name: str):
    _dbg(f"\n[DEBUG] {name}: shape={df.shape}")
    _dbg(f"[DEBUG] {name}: columns={list(df.columns)}")
    # show head but truncate to 3 rows and first 8 cols to avoid spam
    _dbg(f"[DEBUG] {name}: head=\n{df.iloc[:3, :8]}")

def _safe_drop_unnamed(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = list(df.columns)
    df2 = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()
    after = list(df2.columns)
    if before != after:
        _dbg(f"[DEBUG] Dropped Unnamed cols in {name}: {set(before)-set(after)}")
    return df2

def _guess_col_verbose(df: pd.DataFrame, candidates: list[str], label: str) -> str | None:
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}
    # exact case-insensitive match
    for cand in candidates:
        if cand.lower() in lower:
            _dbg(f"[DEBUG] {label}: exact match for {cand!r} -> {lower[cand.lower()]!r}")
            return lower[cand.lower()]
    # substring fallback
    for c in cols:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                _dbg(f"[DEBUG] {label}: substring match for {cand!r} -> {c!r}")
                return c
    _dbg(f"[DEBUG] {label}: NO match among {candidates}")
    return None

def _standardize_hist_debug(df: pd.DataFrame, label_alias: str, run_kind: str) -> pd.DataFrame:
    """
    Verbose version of _standardize_hist to catch header/shape issues early.
    Normalizes a history CSV to have at least: ['epoch','total','ce','run_label','run_kind','run_id'].
    """
    _validate_df_for_hist(df, f"RAW-{label_alias}")

    df = _safe_drop_unnamed(df.copy(), name=label_alias)

    epoch_col = _guess_col_verbose(df, ["epoch","epochs","Epoch"], f"{label_alias}/epoch")
    if epoch_col is None:
        raise ValueError(f"{label_alias}: could not find an epoch column; columns={list(df.columns)}")

    total_col = _guess_col_verbose(df, ["total", "total_loss", "loss", "Avg Total Loss", "Avg Total"],
                                   f"{label_alias}/total")
    if total_col is None:
        raise ValueError(f"{label_alias}: could not find a total/overall loss column; columns={list(df.columns)}")

    ce_col = _guess_col_verbose(df, ["ce", "cross_entropy", "Avg CE Loss", "ce_loss"],
                                f"{label_alias}/ce (optional)")

    # Avoid any "length mismatch" by using rename(), not reassigning df.columns
    if epoch_col != "epoch":
        df.rename(columns={epoch_col: "epoch"}, inplace=True)
    if total_col != "total":
        df.rename(columns={total_col: "total"}, inplace=True)
    if ce_col and ce_col != "ce":
        df.rename(columns={ce_col: "ce"}, inplace=True)
    elif "ce" not in df.columns:
        df["ce"] = np.nan  # add missing ce as NaN

    # coerce numerics
    for c in ("epoch", "total", "ce"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # ensure run_id exists (seed/trial/fold or fallback)
    run_id_found = None
    for k in ["run_id", "seed", "trial", "fold"]:
        if k in df.columns:
            df["run_id"] = df[k].astype(str)
            run_id_found = k
            break
    if run_id_found is None:
        df["run_id"] = run_kind  # single-run fallback
        _dbg(f"[DEBUG] {label_alias}: no run_id/seed/trial/fold; using fallback run_id='{run_kind}'")

    df["run_label"] = label_alias
    df["run_kind"]  = run_kind

    # final sanity
    expected_subset = ["epoch","total","ce","run_label","run_kind","run_id"]
    _dbg(f"[DEBUG] {label_alias}: standardized columns present={ [c for c in expected_subset if c in df.columns] }")
    _validate_df_for_hist(df, f"STD-{label_alias}")
    return df

# ---------- Wrap your dual plot to use the debug standardizer ----------
def plot_pal_vs_ce_dual_from_dfs(
    df_pal_all: pd.DataFrame,
    df_ce_all: pd.DataFrame,
    out_pdf: str,
    logy_total: bool = False,
    inset_epochs: tuple[int,int] | None = (1,3),
    smooth_alpha: float = 0.35
):
    _dbg("\n[DEBUG] Enter plot_pal_vs_ce_dual_from_dfs()")
    _validate_df_for_hist(df_pal_all, "IN-PAL")
    _validate_df_for_hist(df_ce_all,  "IN-CE")

    # Use the debug standardizer so we can see the mappings
    df_pal_all = _standardize_hist_debug(df_pal_all, "With PAL", "pal_run")
    df_ce_all  = _standardize_hist_debug(df_ce_all,  "CE-only",  "ce_run")

    # --- existing logic (unchanged) ---
    def _per_epoch_mean_ci(df: pd.DataFrame, key: str) -> pd.DataFrame:
        if key not in df.columns:
            return pd.DataFrame({"epoch": [], f"{key}_mean": [], f"{key}_ci": []})
        g = df.groupby(["run_id","epoch"], as_index=False)[key].mean()
        agg = g.groupby("epoch")[key].agg(["mean","std","count"]).reset_index()
        ci = 1.96 * (agg["std"] / np.sqrt(agg["count"].clip(lower=1)))
        out = pd.DataFrame({
            "epoch": agg["epoch"].astype(int),
            f"{key}_mean": agg["mean"].astype(float),
            f"{key}_ci":   ci.fillna(0.0).astype(float),
        }).sort_values("epoch").reset_index(drop=True)
        _dbg(f"[DEBUG] per-epoch {key}: shape={out.shape}, head=\n{out.head(3)}")
        return out

    pal_tot = _per_epoch_mean_ci(df_pal_all, "total")
    ce_tot  = _per_epoch_mean_ci(df_ce_all,  "total")
    pal_ce  = _per_epoch_mean_ci(df_pal_all, "ce")
    ce_ce   = _per_epoch_mean_ci(df_ce_all,  "ce")

    # align epochs
    common = np.intersect1d(pal_tot["epoch"].to_numpy(), ce_tot["epoch"].to_numpy())
    _dbg(f"[DEBUG] common epochs = {common.tolist()}")
    pal_tot = pal_tot[pal_tot["epoch"].isin(common)]
    ce_tot  = ce_tot[ce_tot["epoch"].isin(common)]
    pal_ce  = pal_ce[pal_ce["epoch"].isin(common)]
    ce_ce   = ce_ce[ce_ce["epoch"].isin(common)]
    epochs  = pal_tot["epoch"].to_numpy()

    # smooth
    def _ema(x: np.ndarray, alpha: float = smooth_alpha):
        if len(x) == 0: return x
        y = np.zeros_like(x, dtype=float)
        y[0] = x[0]
        for i in range(1, len(x)):
            y[i] = alpha * x[i] + (1 - alpha) * y[i-1]
        return y

    pt_s = _ema(pal_tot["total_mean"].to_numpy())
    ct_s = _ema(ce_tot["total_mean"].to_numpy())
    pc_s = _ema(pal_ce["ce_mean"].to_numpy()) if len(pal_ce) else np.zeros_like(pt_s)
    cc_s = _ema(ce_ce["ce_mean"].to_numpy())  if len(ce_ce)  else np.zeros_like(ct_s)

    _dbg(f"[DEBUG] epochs={epochs.tolist()}")
    _dbg(f"[DEBUG] pt_s (total, PAL) len={len(pt_s)}, ct_s (total, CE) len={len(ct_s)}")
    _dbg(f"[DEBUG] pc_s (ce, PAL) len={len(pc_s)},  cc_s (ce, CE) len={len(cc_s)}")

    # --- plotting (unchanged) ---
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
    fig, (ax1, ax2) = plt.subplots(2,1, figsize=(10,7.2), dpi=140, sharex=True)
    fig.subplots_adjust(hspace=0.18)

    ax1.grid(True, alpha=0.25, linestyle="--", linewidth=0.7)
    ax1.plot(epochs, pt_s, linewidth=2.5, label="With PAL (total, smoothed)")
    ax1.plot(epochs, ct_s, linewidth=2.5, linestyle="--", label="CE-only (total, smoothed)")
    if pal_tot.get("total_ci", pd.Series()).gt(0).any():
        ax1.fill_between(epochs, pal_tot["total_mean"]-pal_tot["total_ci"], pal_tot["total_mean"]+pal_tot["total_ci"], alpha=0.10)
    if ce_tot.get("total_ci", pd.Series()).gt(0).any():
        ax1.fill_between(epochs, ce_tot["total_mean"]-ce_tot["total_ci"], ce_tot["total_mean"]+ce_tot["total_ci"], alpha=0.10)
    ax1.fill_between(epochs, np.minimum(pt_s, ct_s), np.maximum(pt_s, ct_s), alpha=0.12)
    delta = ct_s[-1] - pt_s[-1] if len(pt_s) and len(ct_s) else 0.0
    if len(epochs):
        ax1.scatter([epochs[-1], epochs[-1]], [pt_s[-1], ct_s[-1]], s=35, zorder=3)
        ax1.annotate(f"Δ @ epoch {int(epochs[-1])}: {delta:.3f}",
                     xy=(epochs[-1], pt_s[-1] if len(pt_s) else 0.0),
                     xytext=(epochs[-1], max(pt_s.max() if len(pt_s) else 0.0,
                                             ct_s.max() if len(ct_s) else 0.0)*1.03),
                     ha="right", arrowprops=dict(arrowstyle="->", lw=1.0))
    if logy_total:
        ax1.set_yscale("log")
    ax1.set_ylabel("Training loss" + (" (log)" if logy_total else ""))
    ax1.set_title("PAL vs. CE-only (Bengali Captioning)", pad=10)
    ax1.legend(loc="upper right")

    # inset
    if inset_epochs is not None and len(epochs):
        e1, e2 = inset_epochs
        mask = (epochs>=e1) & (epochs<=e2)
        if mask.any():
            axins = inset_axes(ax1, width="38%", height="45%", loc="center right", borderpad=1.5)
            axins.plot(epochs[mask], pt_s[mask], linewidth=2.0)
            axins.plot(epochs[mask], ct_s[mask], linewidth=2.0, linestyle="--")
            axins.set_xlim(e1, e2)
            axins.grid(True, alpha=0.2, linestyle="--", linewidth=0.6)
            mark_inset(ax1, axins, loc1=2, loc2=4, fc="none", ec="0.5")

    # bottom panel
    ax2.grid(True, alpha=0.25, linestyle="--", linewidth=0.7)
    ax2.plot(epochs, pc_s, linewidth=2.5, label="With PAL (CE component, smoothed)")
    ax2.plot(epochs, cc_s, linewidth=2.5, linestyle="--", label="CE-only (CE component, smoothed)")
    if "ce_mean" in pal_ce and "ce_ci" in pal_ce:
        ax2.fill_between(epochs, pal_ce["ce_mean"]-pal_ce["ce_ci"], pal_ce["ce_mean"]+pal_ce["ce_ci"], alpha=0.10)
    if "ce_mean" in ce_ce and "ce_ci" in ce_ce:
        ax2.fill_between(epochs, ce_ce["ce_mean"]-ce_ce["ce_ci"], ce_ce["ce_mean"]+ce_ce["ce_ci"], alpha=0.10)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Cross-entropy loss")
    ax2.legend(loc="upper right")

    txt = ("Loss curves showing the effect of Patch Alignment Loss (PAL) during training.\n"
           "Top: total training objective; Bottom: CE-only (apples-to-apples). Bands: 95% CI across runs.")
    bbox = FancyBboxPatch((0.02, -0.33), 0.96, 0.25, transform=ax2.transAxes,
                          boxstyle="round,pad=0.5", linewidth=0.8, alpha=0.05)
    ax2.add_patch(bbox)
    ax2.text(0.03, -0.20, txt, transform=ax2.transAxes, va="top")

    os.makedirs(os.path.dirname(out_pdf) or ".", exist_ok=True)
    if not out_pdf.lower().endswith(".pdf"):
        out_pdf += ".pdf"
    
    # NEW: make sure a renderer exists (inset_axes needs it)
    _ensure_canvas(fig)
    
    # Try tight; if backend complains, fall back gracefully
    try:
        fig.savefig(out_pdf, bbox_inches="tight")
    except Exception as e:
        print(f"[WARN] tight save failed ({e}); retrying without tight bbox.")
        fig.savefig(out_pdf)
    
    print(f"🖼️ Saved dual comparison (PDF): {out_pdf}")
    
    # Avoid IPython auto-render attempting to access a missing renderer
    plt.close(fig)


In [ ]:
%env CE_ONLY_CKPT_IN=/kaggle/input/ckpt/transformers/default/1/ce_only.pt
%env WITH_PAL_CKPT_IN=/kaggle/input/ckpt/transformers/default/1/with_pal.pt
%env EMBED_METHOD=umap
%env EMBED_MAX_BATCHES=0
%env EMBED_PDF_PATH=/kaggle/working/embeddings_ce_vs_pal.pdf

In [ ]:
# ============================
# Multimodal Embedding Viz: Real vs Synthetic (CE-only vs PAL)
# ============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.backends.backend_pdf import PdfPages

from math import sqrt
import os, sys, warnings

# UMAP → preferred; fallback to t-SNE
try:
    import umap.umap_ as umap
    _HAS_UMAP = True
except Exception:
    from sklearn.manifold import TSNE
    _HAS_UMAP = False

# ---------- CONFIG via env (override at run time) ----------
EMBED_MAX_BATCHES   = int(os.getenv("EMBED_MAX_BATCHES", "0"))   # 0 or <0 = use ALL batches
N_PLOT_MAX          = int(os.getenv("N_PLOT_MAX", "4000"))       # downsample for plotting if too many points
EMBED_METHOD        = os.getenv("EMBED_METHOD", "umap").lower()
EMBED_RANDOM_STATE  = int(os.getenv("EMBED_SEED", os.getenv("SEED", "42")))

EMBED_PDF_PATH      = os.getenv("EMBED_PDF_PATH", "/kaggle/working/embeddings_ce_vs_pal.pdf")

# **Provide these two when visualizing** (full checkpoints saved by your code):
CE_ONLY_CKPT_IN     = os.getenv("CE_ONLY_CKPT_IN",  "/kaggle/input/ckpt/transformers/default/1/ce_only.pt")  # e.g., /kaggle/input/ckpts/ce_only.ckpt
WITH_PAL_CKPT_IN    = os.getenv("WITH_PAL_CKPT_IN", "/kaggle/input/ckpt/transformers/default/1/with_pal.pt")  # e.g., /kaggle/input/ckpts/with_pal.ckpt

# ---------- helpers ----------
def _gaussian_ci_ellipse(points_2d: np.ndarray, level: float = 0.95):
    """
    Return (width, height, angle_degrees) of the covariance ellipse.
    points_2d: (N,2)
    """
    if len(points_2d) < 2:
        return 0, 0, 0
    cov = np.cov(points_2d.T)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    # 95% CI scale (chi-square with df=2) ≈ 5.991
    chi2_val = 5.991
    width, height = 2 * np.sqrt(vals * chi2_val)
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    return width, height, angle

def _mmd_rbf(X: np.ndarray, Y: np.ndarray, gamma: float | None = None) -> float:
    """Simple unbiased MMD with RBF kernel."""
    def _rbf(a, b):
        # ||a-b||^2
        aa = np.sum(a*a, axis=1, keepdims=True)
        bb = np.sum(b*b, axis=1, keepdims=True)
        ab = a @ b.T
        d2 = aa - 2*ab + bb.T
        if gamma is None:
            # median heuristic on concat
            med = np.median(d2[d2>0]) if np.any(d2>0) else 1.0
            g = 1.0 / (2.0 * med)
        else:
            g = gamma
        return np.exp(-g * d2)
    n = X.shape[0]; m = Y.shape[0]
    if n<2 or m<2:
        return 0.0
    Kxx = _rbf(X, X); np.fill_diagonal(Kxx, 0.0)
    Kyy = _rbf(Y, Y); np.fill_diagonal(Kyy, 0.0)
    Kxy = _rbf(X, Y)
    mmd = (Kxx.sum()/(n*(n-1))) + (Kyy.sum()/(m*(m-1))) - (2.0*Kxy.mean())
    return float(mmd)

def _reduce_2d(Z: np.ndarray, method: str = "umap", random_state: int = 42) -> np.ndarray:
    method = method.lower()
    if method == "umap" and _HAS_UMAP:
        reducer = umap.UMAP(
            n_components=2, n_neighbors=20, min_dist=0.1, metric="cosine",
            random_state=random_state
        )
        return reducer.fit_transform(Z)
    else:
        if method == "umap":
            warnings.warn("UMAP not installed; falling back to t-SNE.")
        reducer = TSNE(
            n_components=2, perplexity=30, learning_rate="auto",
            init="pca", random_state=random_state, n_iter=1000
        )
        return reducer.fit_transform(Z)

@torch.no_grad()
def _batch_to_encoder_tokens(model, real_pixel_values, synthetic_pixel_values):
    """Encode last-scale features and project to token space (B,S,D)."""
    # model._encode_feats_list returns list of feature maps
    real_last  = model._encode_feats_list(real_pixel_values)[-1]
    synth_last = model._encode_feats_list(synthetic_pixel_values)[-1]
    real_tok, _  = model._project_norm(real_last,  use_prev=False)  # (B,S,D)
    synth_tok, _ = model._project_norm(synth_last, use_prev=False)  # (B,S,D)
    # Ensure decoder dtype consistency
    real_tok  = real_tok.to(model.dec_dtype)
    synth_tok = synth_tok.to(model.dec_dtype)
    return real_tok, synth_tok

@torch.no_grad()
def _pool_encoder_tokens_avg(tokens: torch.Tensor) -> torch.Tensor:
    """Global average pool over spatial tokens: (B,S,D) -> (B,D)."""
    return tokens.mean(dim=1)

@torch.no_grad()
def _pool_tokens_pal_weighted(model, real_tok, synth_tok, labels, attn_mask):
    """
    Use decoder cross-attention to get patch weights (as in PAL),
    then weighted pool tokens to (B,D). If cross-attn is not available,
    falls back to uniform averaging.
    """
    B, S, D = real_tok.shape
    # Run decoder forward for attention maps (teacher-forced)
    enc_out = BaseModelOutput(last_hidden_state=real_tok)
    dec_out = model.language_decoder(
        encoder_outputs=enc_out,
        labels=labels,
        decoder_attention_mask=attn_mask,
        output_attentions=True,
        return_dict=True
    )
    cross = dec_out.cross_attentions
    if cross is None or all(a is None for a in cross):
        # fallback
        return _pool_encoder_tokens_avg(real_tok), _pool_encoder_tokens_avg(synth_tok)

    # weights from cross-attn
    w = model._attention_to_patch_weights(cross, attn_mask, S)  # (B,S)
    w = w.float().unsqueeze(-1)  # (B,S,1)
    r_pool = (real_tok.float() * w).sum(dim=1)
    s_pool = (synth_tok.float() * w).sum(dim=1)
    return r_pool.to(real_tok.dtype), s_pool.to(synth_tok.dtype)

def _collect_embeddings(model, dataloader, device, max_batches=0, use_pal_weights=False):
    """Return two numpy arrays: real_embs (N,D) and synth_embs (N,D)."""
    real_list, synth_list = [], []
    model.eval()
    for batch_idx, batch in enumerate(dataloader):
        if batch is None:
            continue
        real_pixel_values      = batch["real_pixel_values"].to(device, non_blocking=True)
        synthetic_pixel_values = batch["synthetic_pixel_values"].to(device, non_blocking=True)
        labels    = batch.get("labels", None)
        attn_mask = batch.get("attention_mask", None)

        rtok, stok = _batch_to_encoder_tokens(model, real_pixel_values, synthetic_pixel_values)
        if use_pal_weights and (labels is not None) and (attn_mask is not None):
            labels    = labels.to(device, non_blocking=True)
            attn_mask = attn_mask.to(device, non_blocking=True)
            rpool, spool = _pool_tokens_pal_weighted(model, rtok, stok, labels, attn_mask)
        else:
            rpool = _pool_encoder_tokens_avg(rtok)
            spool = _pool_encoder_tokens_avg(stok)

        real_list.append(rpool.detach().cpu().numpy())
        synth_list.append(spool.detach().cpu().numpy())

        # stop only if max_batches > 0
        if max_batches and (batch_idx + 1) >= max_batches:
            break

    if len(real_list) == 0:
        return np.zeros((0, model.d_model)), np.zeros((0, model.d_model))

    real  = np.concatenate(real_list,  axis=0)
    synth = np.concatenate(synth_list, axis=0)

    # optional: uniform subsample to keep the figure readable
    if N_PLOT_MAX > 0 and real.shape[0] > N_PLOT_MAX:
        rng = np.random.default_rng(EMBED_RANDOM_STATE)
        idx = rng.choice(real.shape[0], size=N_PLOT_MAX, replace=False)
        real, synth = real[idx], synth[idx]
    return real, synth


def _panel(ax, Xr2, Xs2, title, colors=("tab:blue","tab:orange")):
    ax.grid(True, alpha=0.25, linestyle="--", linewidth=0.6)
    ax.scatter(Xr2[:,0], Xr2[:,1], s=8,  alpha=0.35, label="Real",      color=colors[0])
    ax.scatter(Xs2[:,0], Xs2[:,1], s=8,  alpha=0.35, label="Synthetic", color=colors[1])

    # centroids
    cr = Xr2.mean(axis=0); cs = Xs2.mean(axis=0)
    ax.scatter([cr[0]],[cr[1]], s=80, marker="X", color=colors[0], edgecolor="k", zorder=5)
    ax.scatter([cs[0]],[cs[1]], s=80, marker="X", color=colors[1], edgecolor="k", zorder=5)
    ax.plot([cr[0], cs[0]], [cr[1], cs[1]], color="k", linewidth=1.0, alpha=0.6)

    # ellipses
    wr, hr, ar = _gaussian_ci_ellipse(Xr2, 0.95)
    ws, hs, as_ = _gaussian_ci_ellipse(Xs2, 0.95)
    er = Ellipse(xy=cr, width=wr, height=hr, angle=ar, fill=False, lw=1.6, color=colors[0], alpha=0.9)
    es = Ellipse(xy=cs, width=ws, height=hs, angle=as_, fill=False, lw=1.6, color=colors[1], alpha=0.9)
    ax.add_patch(er); ax.add_patch(es)

    # metrics
    cdist = float(np.linalg.norm(cr - cs))
    mmd   = _mmd_rbf(Xr2, Xs2)

    ax.set_title(f"{title}\nCentroid Δ={cdist:.3f}  |  MMD={mmd:.3f}", fontsize=11.5, pad=10)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc="best", frameon=True)

def _build_and_load_for_eval(ckpt_path: str, device):
    """Instantiate your model and load a full checkpoint or raw state dict."""
    model = MaxVitMbartCaptioningModel(
        MAXVIT_MODEL_NAME,
        MBART_MODEL_NAME,
        bn_in_token_id=tokenizer.lang_code_to_id.get("bn_IN"),
        patch_alignment_weight=INITIAL_PATCH_ALIGNMENT_WEIGHT,
        attn_temp=PAL_CFG["attn_temp"],
        topk_ratio=PAL_CFG["topk_ratio"],
        use_last_k_cross_layers=PAL_CFG["use_last_k_cross_layers"],
        pool_factor=PAL_CFG["pool_factor"],
        multi_scale=PAL_CFG["multi_scale"],
        use_info_nce=USE_INFO_NCE,
        info_nce_temp=INFO_NCE_CFG["temp"],
        info_nce_coef=INFO_NCE_CFG["coef"],
        use_ot=USE_OT,
        ot_reg=OT_CFG["reg"],
        ot_iters=OT_CFG["iters"],
        ot_topk_ratio=OT_CFG["topk_ratio"],
        ot_coef=OT_CFG["coef"]
    ).to(device)
    model.eval()

    if not ckpt_path or (not os.path.exists(ckpt_path)):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    state = torch.load(ckpt_path, map_location=device)
    # support full checkpoint dict or raw SD
    if isinstance(state, dict) and "model_state_dict" in state:
        missing, unexpected = model.load_state_dict(state["model_state_dict"], strict=False)
    else:
        missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"Loaded {os.path.basename(ckpt_path)} | Missing: {len(missing)} | Unexpected: {len(unexpected)}")
    return model

def visualize_embeddings_ce_vs_pal(
    dataloader_eval,
    ce_ckpt_path: str,
    pal_ckpt_path: str,
    method: str = EMBED_METHOD,
    pdf_path: str = EMBED_PDF_PATH,
    max_batches: int = EMBED_MAX_BATCHES,
    random_state: int = EMBED_RANDOM_STATE
):
    """
    Produces a two-page PDF:
      Page 1: Encoder-only pooled embeddings (CE-only vs PAL)
      Page 2: PAL-weighted pooled embeddings (if labels available)
    """
    if not ce_ckpt_path or not pal_ckpt_path:
        raise ValueError("Please set CE_ONLY_CKPT_IN and WITH_PAL_CKPT_IN to valid checkpoint files.")

    # Load CE model, collect embeddings
    ce_model = _build_and_load_for_eval(ce_ckpt_path, DEVICE)
    real_ce, syn_ce = _collect_embeddings(ce_model, dataloader_eval, DEVICE, max_batches=max_batches, use_pal_weights=False)
    
    # Offload CE model
    del ce_model
    torch.cuda.empty_cache()
    
    # Load PAL model, collect embeddings
    pal_model = _build_and_load_for_eval(pal_ckpt_path, DEVICE)
    real_pal, syn_pal = _collect_embeddings(pal_model, dataloader_eval, DEVICE, max_batches=max_batches, use_pal_weights=False)
    
    del pal_model
    torch.cuda.empty_cache()


    # reduce-to-2D per panel (fit reducer on each condition's real+syn)
    X_ce  = np.concatenate([real_ce, syn_ce], axis=0)
    X_pal = np.concatenate([real_pal, syn_pal], axis=0)

    Z_ce  = _reduce_2d(X_ce,  method=method, random_state=random_state)
    Z_pal = _reduce_2d(X_pal, method=method, random_state=random_state)

    Zr_ce,  Zs_ce  = Z_ce[:len(real_ce)],  Z_ce[len(real_ce):]
    Zr_pal, Zs_pal = Z_pal[:len(real_pal)], Z_pal[len(real_pal):]

    pdf_path = pdf_path if pdf_path.lower().endswith(".pdf") else (pdf_path + ".pdf")
    with PdfPages(pdf_path) as pdf:
        # Page 1
        fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), dpi=140, constrained_layout=True)
        _panel(axes[0], Zr_ce,  Zs_ce,  f"CE-only  |  {method.upper()}")
        _panel(axes[1], Zr_pal, Zs_pal, f"With PAL  |  {method.upper()}")
        fig.suptitle("Encoder-only embeddings: Real vs Synthetic", y=1.04, fontsize=13)
        # caption
        fig.text(0.02, -0.02, "t-SNE/UMAP visualization of real vs synthetic image embeddings (after vision encoder).", fontsize=10)
        pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)

        # ---- B) PAL-weighted pooling (requires labels in loader) ----
        try:
            real_ce_w,  syn_ce_w  = _collect_embeddings(ce_model,  dataloader_eval, DEVICE, max_batches=max_batches, use_pal_weights=True)
            real_pal_w, syn_pal_w = _collect_embeddings(pal_model, dataloader_eval, DEVICE, max_batches=max_batches, use_pal_weights=True)

            if real_ce_w.shape[0] > 0 and real_pal_w.shape[0] > 0:
                X_cew  = np.concatenate([real_ce_w, syn_ce_w], axis=0)
                X_paw  = np.concatenate([real_pal_w, syn_pal_w], axis=0)
                Z_cew  = _reduce_2d(X_cew, method=method, random_state=random_state)
                Z_paw  = _reduce_2d(X_paw, method=method, random_state=random_state)
                Zr_cew, Zs_cew  = Z_cew[:len(real_ce_w)],  Z_cew[len(real_ce_w):]
                Zr_paw, Zs_paw  = Z_paw[:len(real_pal_w)], Z_paw[len(real_pal_w):]

                fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5.2), dpi=140, constrained_layout=True)
                _panel(axes2[0], Zr_cew,  Zs_cew,  f"CE-only  |  {method.upper()}  (PAL-weighted pooling)")
                _panel(axes2[1], Zr_paw, Zs_paw, f"With PAL  |  {method.upper()}  (PAL-weighted pooling)")
                fig2.suptitle("PAL-weighted embeddings: Real vs Synthetic", y=1.04, fontsize=13)
                fig2.text(0.02, -0.02, "Embeddings pooled using cross-attention patch weights (PAL).", fontsize=10)
                pdf.savefig(fig2, bbox_inches="tight"); plt.close(fig2)
            else:
                warnings.warn("Skipping PAL-weighted page: labels/attn_mask not available in dataloader.")
        except Exception as e:
            warnings.warn(f"Skipping PAL-weighted page due to error: {e}")

    print(f"Saved embedding visualization PDF → {pdf_path}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from transformers import AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput


def _l2n(x, dim=-1, eps=1e-8):
    """L2-normalize along dim with numerical guard."""
    return x / (x.norm(dim=dim, keepdim=True).clamp_min(eps))


class MaxVitMbartCaptioningModel(nn.Module):
    def __init__(
        self,
        encoder_model_name,
        decoder_model_name,
        bn_in_token_id,
        patch_alignment_weight=0.05,

        # == A: stability knobs ==
        attn_temp: float = 1.0,            
        topk_ratio: float = 0.30,         
        use_last_k_cross_layers: int = 2,  

        # efficiency
        pool_factor: int = 1,           

        # == B: InfoNCE ==
        use_info_nce: bool = False,
        info_nce_temp: float = 0.07,
        info_nce_coef: float = 0.3,        # scaled by patch_alignment_weight

        # == C: OT/Sinkhorn ==
        use_ot: bool = False,
        ot_reg: float = 0.05,              # entropic regularization (epsilon)
        ot_iters: int = 50,
        ot_topk_ratio: float = 0.15,       # sparsify patches before OT
        ot_coef: float = 0.5,              # scaled by patch_alignment_weight

        # == E: multi-scale PAL (optional) ==
        multi_scale: bool = False
    ):
        super().__init__()

        # ---------------------------
        # Vision encoder (frozen)
        # ---------------------------
        self.vision_encoder = timm.create_model(
            encoder_model_name, pretrained=True, features_only=True
        )
        self.vision_encoder.eval()
        for p in self.vision_encoder.parameters():
            p.requires_grad = False
        if hasattr(self.vision_encoder, "set_grad_checkpointing"):
            self.vision_encoder.set_grad_checkpointing(True)

        ch_list = self.vision_encoder.feature_info.channels()
        self.last_ch = ch_list[-1]
        self.prev_ch = ch_list[-2] if len(ch_list) >= 2 else None

        # ------------------------------------------
        # Language decoder (defines the "canonical"
        # dtype we keep for projections/LayerNorm)
        # ------------------------------------------
        self.language_decoder = AutoModelForSeq2SeqLM.from_pretrained(
            decoder_model_name, attn_implementation="eager", low_cpu_mem_usage=True
        )
        self.language_decoder.config.decoder_start_token_id = bn_in_token_id
        self.language_decoder.config.use_cache = False
        if hasattr(self.language_decoder, "gradient_checkpointing_enable"):
            self.language_decoder.gradient_checkpointing_enable()

        self.d_model = self.language_decoder.config.d_model
        # decoder param dtype (usually float32 for mBART)
        self.dec_dtype = next(self.language_decoder.parameters()).dtype

        # Projections + LN must match decoder dtype to avoid Half/Float mismatches under AMP
        self.vision_projection = nn.Linear(self.last_ch, self.d_model).to(self.dec_dtype)
        self.encoder_norm = nn.LayerNorm(self.d_model, eps=1e-5).to(self.dec_dtype)
        nn.init.xavier_uniform_(self.vision_projection.weight)
        if self.vision_projection.bias is not None:
            nn.init.zeros_(self.vision_projection.bias)

        # Optional multi-scale projection for the penultimate feature map
        self.multi_scale = bool(multi_scale and self.prev_ch is not None)
        if self.multi_scale:
            self.vision_projection_prev = nn.Linear(self.prev_ch, self.d_model).to(self.dec_dtype)
            nn.init.xavier_uniform_(self.vision_projection_prev.weight)
            if self.vision_projection_prev.bias is not None:
                nn.init.zeros_(self.vision_projection_prev.bias)

        # knobs
        self.patch_alignment_weight = float(patch_alignment_weight)
        self.attn_temp = float(attn_temp)
        self.topk_ratio = float(topk_ratio)
        self.use_last_k_cross_layers = int(max(1, use_last_k_cross_layers))
        self.pool_factor = int(max(1, pool_factor))

        # InfoNCE
        self.use_info_nce = bool(use_info_nce)
        self.info_nce_temp = float(info_nce_temp)
        self.info_nce_coef = float(info_nce_coef)

        # OT
        self.use_ot = bool(use_ot)
        self.ot_reg = float(ot_reg)
        self.ot_iters = int(ot_iters)
        self.ot_topk_ratio = float(ot_topk_ratio)
        self.ot_coef = float(ot_coef)

    # =========================
    # Encoding helpers
    # =========================
    def _encode_feats_list(self, pixel_values):
        """Return [prev?, last] feature maps; optionally downsampled by pool_factor."""
        with torch.no_grad():
            feats_list = self.vision_encoder(pixel_values)     
        outs = []

        idxs = [-1] if not self.multi_scale else [-2, -1]
        for i in idxs:
            f = feats_list[i]                                
            if self.pool_factor > 1:
                H, W = f.shape[-2:]
                f = F.adaptive_avg_pool2d(
                    f, (max(1, H // self.pool_factor), max(1, W // self.pool_factor))
                )
            outs.append(f)
        return outs 

    def _project_norm(self, fmap, use_prev: bool = False):
        """Project (B,C,H,W) -> (B,S,D) with dtype aligned to projection weights + LN dtype."""
        B, C, H, W = fmap.shape
        S = H * W
        patches = fmap.reshape(B, C, S).transpose(1, 2)        # (B,S,C)

        # Choose the right projection head
        proj = self.vision_projection_prev if use_prev else self.vision_projection

        # *** CRITICAL: cast inputs to the Linear's dtype to avoid Half/Float mismatch ***
        patches = patches.to(dtype=proj.weight.dtype)

        enc = proj(patches)                                    # (B,S,D)
        # Keep LN dtype consistent with its parameters (same as decoder dtype)
        enc = self.encoder_norm(enc.to(self.encoder_norm.weight.dtype))
        return enc, (H, W)

    # =========================
    # Cross-attn -> patch weights
    # =========================
    def _attention_to_patch_weights(self, cross_attentions, attention_mask, S):
        """Average last-K cross-attn maps -> (B,S) weights with temp + top-k sparsity."""
        keep = cross_attentions[-self.use_last_k_cross_layers:]
        attn = [torch.nan_to_num(a.float(), 0.0, 0.0, 0.0) for a in keep if a is not None]
        if not attn:
            B = attention_mask.size(0) if attention_mask is not None else 1
            device = (
                attention_mask.device
                if attention_mask is not None
                else next(self.language_decoder.parameters()).device
            )
            return torch.full((B, S), 1.0 / S, device=device)

        cross = torch.stack(attn, dim=0)  # (L,B,H,T,S)

        # Weighted average over T using attention_mask
        if attention_mask is not None:
            T = cross.size(3)
            tgt = attention_mask[:, :T].to(cross.dtype).view(1, -1, 1, T, 1)  # (1,B,1,T,1)
            cross = cross * tgt
            denom = tgt.sum(dim=3, keepdim=True).clamp_min(1.0)
            cross = cross.sum(dim=3, keepdim=True) / denom
            cross = cross.squeeze(3)                      # (L,B,H,S)
        else:
            cross = cross.mean(dim=3)

        # avg heads -> (L,B,S), then layers -> (B,S)
        w = cross.mean(dim=2).mean(dim=0)                 # (B,S)

        # temperature + softmax
        w = torch.softmax(w / max(1e-6, self.attn_temp), dim=-1)

        # sparsify (top-k) to reduce background noise
        if 0.0 < self.topk_ratio < 1.0:
            k = max(1, min(int(self.topk_ratio * w.size(1)), w.size(1)))
            topv, topi = torch.topk(w, k, dim=1)
            mask = torch.zeros_like(w).scatter_(1, topi, 1.0)
            w = w * mask
            w = w / w.sum(dim=1, keepdim=True).clamp_min(1e-8)

        bad = ~torch.isfinite(w).all(dim=1, keepdim=True)
        if bad.any():
            uniform = torch.full_like(w, 1.0 / w.size(1))
            w = torch.where(bad, uniform, w)
        return w  # (B,S)

    # =========================
    # Sinkhorn OT (per-sample)
    # =========================
    def _sinkhorn_cost(self, r, s, wr, ws, eps=0.05, iters=50):
        """
        r,s: (Sr,D) and (Ss,D) L2-normalized
        wr,ws: (Sr,), (Ss,) masses sum to 1
        returns scalar cost ~ <T, C> with C = 1 - r @ s^T
        """
        # cos in [-1,1]; (cos-1)/eps <= 0 -> K is well-conditioned for small eps
        K = torch.exp((r @ s.t() - 1.0) / eps)
        u = torch.ones_like(wr) / wr.numel()
        v = torch.ones_like(ws) / ws.numel()

        for _ in range(iters):
            u = wr / (K @ v).clamp_min(1e-12)
            v = ws / (K.t() @ u).clamp_min(1e-12)

        T = torch.diag(u) @ K @ torch.diag(v)            # (Sr,Ss)
        C = 1.0 - (r @ s.t())
        return (T * C).sum()                              # scalar

    # =========================
    # Forward
    # =========================
    def forward(self, real_pixel_values, synthetic_pixel_values, labels=None, attention_mask=None):
        # --- Encode real + synthetic (multi-scale aware) ---
        real_feats_list  = self._encode_feats_list(real_pixel_values)
        synth_feats_list = self._encode_feats_list(synthetic_pixel_values)

        if self.multi_scale:
            real_prev, real_last   = real_feats_list
            synth_prev, synth_last = synth_feats_list
        else:
            real_last   = real_feats_list[-1]
            synth_last  = synth_feats_list[-1]

        real_proj_last, (H_l, W_l) = self._project_norm(real_last, use_prev=False)
        synth_proj_last, _          = self._project_norm(synth_last, use_prev=False)
        S_last = H_l * W_l

        # *** Ensure encoder outputs match decoder dtype ***
        real_proj_last = real_proj_last.to(self.dec_dtype)

        # CE via decoder
        enc_out = BaseModelOutput(last_hidden_state=real_proj_last)
        dec_out = self.language_decoder(
            encoder_outputs=enc_out,
            labels=labels,
            decoder_attention_mask=attention_mask,
            output_attentions=True,
            return_dict=True
        )
        ce = dec_out.loss

        # patch weights from cross-attn (no grad)
        with torch.no_grad():
            w_last = self._attention_to_patch_weights(dec_out.cross_attentions, attention_mask, S_last)  # (B,S_last)

        # --- PAL on last scale (fp32) ---
        with torch.amp.autocast('cuda', enabled=False):
            r = real_proj_last.float()
            s = synth_proj_last.float()
            w = w_last.float().unsqueeze(-1)
            r_pool = (r * w).sum(dim=1)
            s_pool = (s * w).sum(dim=1)
            sim = F.cosine_similarity(r_pool, s_pool, dim=-1, eps=1e-8).clamp(-1.0, 1.0)
            pal_last = (1.0 - sim).mean()
            total_pal = pal_last

            # multi-scale PAL: align the penultimate stage too
            if self.multi_scale:
                real_proj_prev, (H_p, W_p) = self._project_norm(real_prev, use_prev=True)
                synth_proj_prev, _ = self._project_norm(synth_prev, use_prev=True)

                # upsample weights from (H_l, W_l) -> (H_p, W_p)
                w_grid = w_last.view(w_last.size(0), 1, H_l, W_l)
                w_prev = F.interpolate(
                    w_grid, size=(H_p, W_p), mode='bilinear', align_corners=False
                ).flatten(2).squeeze(1)
                w_prev = w_prev / w_prev.sum(dim=1, keepdim=True).clamp_min(1e-8)

                rp = real_proj_prev.float()
                sp = synth_proj_prev.float()
                wp = w_prev.float().unsqueeze(-1)
                rp_pool = (rp * wp).sum(dim=1)
                sp_pool = (sp * wp).sum(dim=1)
                sim_p = F.cosine_similarity(rp_pool, sp_pool, dim=-1, eps=1e-8).clamp(-1.0, 1.0)
                pal_prev = (1.0 - sim_p).mean()
                total_pal = 0.5 * (pal_last + pal_prev)

            # InfoNCE
            nce = r_pool.new_tensor(0.0)
            if self.use_info_nce and r_pool.size(0) >= 2:
                r_n = _l2n(r_pool)
                s_n = _l2n(s_pool)
                feats = torch.cat([r_n, s_n], dim=0)               # (2B,D)
                logits = (feats @ feats.t()) / max(1e-6, self.info_nce_temp)
                logits.fill_diagonal_(-1e9)
                idx = torch.arange(feats.size(0), device=feats.device)
                pos = idx ^ (feats.size(0)//2)                     # i <-> i^B
                nce = -F.log_softmax(logits, dim=1)[idx, pos].mean()

            # OT/Sinkhorn (per-sample loop)
            ot_cost = r_pool.new_tensor(0.0)
            if self.use_ot:
                B, S = r.size(0), r.size(1)
                if 0.0 < self.ot_topk_ratio < 1.0:
                    k = max(1, min(int(self.ot_topk_ratio * S), S))
                    wv, wi = torch.topk(w_last, k, dim=1)
                else:
                    k = S
                    wi = torch.arange(S, device=r.device).unsqueeze(0).expand(B, -1)
                    wv = w_last
                for b in range(B):
                    ridx = wi[b]
                    sidx = wi[b]  # symmetric topk on synth too (proxy)
                    rnorm = _l2n(r[b, ridx, :], dim=-1)
                    snorm = _l2n(s[b, sidx, :], dim=-1)
                    wr = (wv[b] / wv[b].sum()).detach()
                    ws = (wv[b] / wv[b].sum()).detach()
                    ot_cost = ot_cost + self._sinkhorn_cost(
                        rnorm, snorm, wr, ws, eps=self.ot_reg, iters=self.ot_iters
                    )
                ot_cost = ot_cost / B

        total = ce + self.patch_alignment_weight * total_pal
        if self.use_info_nce:
            total = total + (self.info_nce_coef * self.patch_alignment_weight) * nce
        if self.use_ot:
            total = total + (self.ot_coef * self.patch_alignment_weight) * ot_cost

        return {
            "loss": total,
            "cross_entropy_loss": ce,
            "patch_alignment_loss": total_pal,
            "info_nce_loss": (nce if self.use_info_nce else None),
            "ot_loss": (ot_cost if self.use_ot else None),
            "logits": dec_out.logits
        }

    # =========================
    # Generation
    # =========================
    @torch.no_grad()
    def generate(
        self,
        real_pixel_values,
        tokenizer,
        max_new_tokens: int = 128,
        num_beams: int = 6,
        num_return_sequences: int = None,  # default None → auto set to num_beams
        length_penalty: float = 1.3,
        no_repeat_ngram_size: int = 3,
        min_new_tokens: int = 16,
        repetition_penalty: float = 1.02
    ):
        self.eval()
        # encode only last scale for decoding
        last = self._encode_feats_list(real_pixel_values)[-1]
        enc_last, _ = self._project_norm(last, use_prev=False)
    
        # match decoder dtype
        dec_dtype = self.dec_dtype
        enc_last = enc_last.to(dtype=dec_dtype)
    
        encoder_outputs = BaseModelOutput(last_hidden_state=enc_last)
    
        bos_id = tokenizer.lang_code_to_id.get("bn_IN")
        if bos_id is None:
            raise ValueError("bn_IN not found in tokenizer.lang_code_to_id")
        eos_id = tokenizer.eos_token_id
        pad_id = tokenizer.pad_token_id
    
        # if num_return_sequences not specified, generate as many as beams
        if num_return_sequences is None:
            num_return_sequences = num_beams
    
        prev_cache = getattr(self.language_decoder.config, "use_cache", True)
        self.language_decoder.config.use_cache = True
        try:
            out = self.language_decoder.generate(
                encoder_outputs=encoder_outputs,
                decoder_start_token_id=bos_id,
                forced_bos_token_id=bos_id,
                eos_token_id=eos_id,
                pad_token_id=pad_id,
                num_beams=num_beams,
                num_return_sequences=num_return_sequences,
                length_penalty=max(1.0, float(length_penalty)),
                no_repeat_ngram_size=no_repeat_ngram_size,
                repetition_penalty=repetition_penalty,
                early_stopping=True,
                max_new_tokens=max_new_tokens,
                min_new_tokens=min_new_tokens,
                return_dict_in_generate=True
            )
        finally:
            self.language_decoder.config.use_cache = prev_cache
    
        return tokenizer.batch_decode(
            out.sequences, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )


In [ ]:
# =============================
# Incremental training (7 epochs)
# with CE + PA + InfoNCE + OT
# and proper resume (model/opt/sched/scaler)
# =============================
import os, json
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
import timm
from transformers import AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput


# --- Enable-at-epoch switches ---
NCE_ENABLE_AT_EPOCH = 0
OT_ENABLE_AT_EPOCH  = 0

# -----------------------------
# Checkpoint helpers
# -----------------------------
def save_checkpoint(path, model, optimizer, scheduler, scaler):
    ckpt = {
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
    }
    torch.save(ckpt, path)
    print(f"Saved checkpoint to {path}")

def load_checkpoint_if_any(path, device, model, optimizer, scheduler, scaler):
    if not os.path.exists(path):
        print("No previous checkpoint found, training from scratch for this batch.")
        return
    ckpt = torch.load(path, map_location=device)
    # Support both "full" checkpoint and legacy "weights only"
    if "model_state_dict" in ckpt:
        missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
        print(f"Model loaded. Missing keys: {missing} | Unexpected: {unexpected}")
        try:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            scaler.load_state_dict(ckpt["scaler_state_dict"])
            print("Optimizer/Scheduler/Scaler states loaded.")
        except Exception as e:
            print(f"Could not load optimizer/scheduler/scaler states: {e} (will reinitialize them)")
    else:
        missing, unexpected = model.load_state_dict(ckpt, strict=False)
        print(f"Legacy weights loaded. Missing keys: {missing} | Unexpected: {unexpected}")

# -----------------------------
# Progressive unfreezing
# -----------------------------
def unfreeze_layers(model, epoch, total_epochs):
    if epoch >= total_epochs // 3:
        for p in model.language_decoder.model.encoder.layers[0].parameters():
            p.requires_grad = True
    if epoch >= total_epochs // 2:
        for p in model.language_decoder.model.encoder.layers[1].parameters():
            p.requires_grad = True
    if epoch >= 2 * total_epochs // 3:
        for p in model.language_decoder.model.encoder.parameters():
            p.requires_grad = True

# -----------------------------
# Train for exactly 7 epochs
# -----------------------------
def train_for_seven_epochs(
    model, dataloader, optimizer, scheduler, scaler,
    device, grad_accum_steps=GRAD_ACCUM_STEPS, num_epochs=7, history: TrainHistory | None = None
):

    for epoch in range(num_epochs):
        model.train()
        total_epoch_loss = 0.0
        total_ce_loss    = 0.0
        total_pa_loss    = 0.0
        total_nce_loss   = 0.0
        total_ot_loss    = 0.0
        num_batches_processed = 0

        # Unfreeze progressively
        unfreeze_layers(model, epoch, num_epochs)

        optimizer.zero_grad(set_to_none=True)

        for batch_idx, batch in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")):
            if batch is None:
                print(f"Warning: Skipping empty batch {batch_idx+1} in Epoch {epoch+1}.")
                continue

            real_pixel_values      = batch["real_pixel_values"].to(device, non_blocking=True)
            synthetic_pixel_values = batch["synthetic_pixel_values"].to(device, non_blocking=True)
            labels                 = batch["labels"].to(device, non_blocking=True)
            decoder_attention_mask = batch["attention_mask"].to(device, non_blocking=True)

            use_nce = USE_INFO_NCE and (epoch >= NCE_ENABLE_AT_EPOCH)
            use_ot  = USE_OT       and (epoch >= OT_ENABLE_AT_EPOCH)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                outputs = model(
                    real_pixel_values=real_pixel_values,
                    synthetic_pixel_values=synthetic_pixel_values,
                    labels=labels,
                    attention_mask=decoder_attention_mask
                )

                # Base CE loss
                ce_loss = outputs.get("cross_entropy_loss", outputs.get("loss", 0.0))
                pa_loss = outputs.get("patch_alignment_loss", 0.0)
                nce_loss = outputs.get("info_nce_loss", 0.0) if use_nce else 0.0
                ot_loss  = outputs.get("ot_loss", 0.0)       if use_ot  else 0.0

                # Sum of losses
                total_loss = ce_loss + pa_loss
                if use_nce and isinstance(nce_loss, (int, float)) is False:
                    total_loss = total_loss + INFO_NCE_CFG["coef"] * nce_loss
                elif use_nce:
                    total_loss = total_loss + INFO_NCE_CFG["coef"] * nce_loss

                if use_ot and isinstance(ot_loss, (int, float)) is False:
                    total_loss = total_loss + OT_CFG["coef"] * ot_loss
                elif use_ot:
                    total_loss = total_loss + OT_CFG["coef"] * ot_loss

                # Gradient accumulation
                step_loss = total_loss / grad_accum_steps

            # Backward
            if scaler.is_enabled():
                scaler.scale(step_loss).backward()
            else:
                step_loss.backward()

            # Optimizer step
            if (batch_idx + 1) % grad_accum_steps == 0:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                    # clip only current trainable params
                    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                    optimizer.step()

                optimizer.zero_grad(set_to_none=True)

            # ---- Accounting (safe to float()) ----
            def _to_float(x):
                try:
                    return float(x)
                except Exception:
                    try:
                        return x.item()
                    except Exception:
                        return 0.0

            total_epoch_loss += _to_float(step_loss.detach())  # per-step (already / grad_accum)
            total_ce_loss    += _to_float(ce_loss)
            total_pa_loss    += _to_float(pa_loss)
            total_nce_loss   += _to_float(nce_loss)
            total_ot_loss    += _to_float(ot_loss)
            num_batches_processed += 1

            # free
            del real_pixel_values, synthetic_pixel_values, labels, decoder_attention_mask, outputs, step_loss
            torch.cuda.empty_cache()

        scheduler.step()

        # Report
        if num_batches_processed > 0:
            avg_loss   = total_epoch_loss / num_batches_processed
            avg_ce     = total_ce_loss    / num_batches_processed
            avg_pa     = total_pa_loss    / num_batches_processed
            avg_nce    = total_nce_loss   / num_batches_processed
            avg_ot     = total_ot_loss    / num_batches_processed

            print(f"Epoch {epoch+1}: "
                  f"Avg Total Loss = {avg_loss:.4f}, "
                  f"Avg CE Loss = {avg_ce:.4f}, "
                  f"Avg PA Loss = {avg_pa:.4f}, "
                  f"Avg InfoNCE Loss = {avg_nce:.4f}, "
                  f"Avg OT Loss = {avg_ot:.4f}")
            
            # ---- Log to history ----
            if history is not None:
                history.log_epoch(
                    epoch=epoch+1,
                    total=avg_loss,
                    ce=avg_ce,
                    pal=avg_pa,
                    nce=avg_nce,
                    ot=avg_ot
                )
                
        else:
            print(f"Epoch {epoch+1}: No batches processed (check data or collate_fn).")

        torch.cuda.empty_cache()

PAL_CFG = {
    "attn_temp": 1.0,
    "topk_ratio": 0.30,
    "use_last_k_cross_layers": 2,
    "pool_factor": 2,
    "multi_scale": True
}

USE_INFO_NCE = False
INFO_NCE_CFG = {
    "temp": 0.07,
    "coef": 0.3
}

USE_OT = False
OT_CFG = {
    "reg": 0.05,
    "iters": 30,
    "topk_ratio": 0.10,
    "coef": 0.5
}


if __name__ == "__main__":
    print("--- Starting Data Preprocessing ---")

    # Load COCO annotations
    with open(COCO_ANNOTATIONS_PATH, 'r') as f:
        coco_data = json.load(f)
    annotations = coco_data['annotations']
    df_mscoco = pd.DataFrame(annotations)[['id', 'image_id', 'caption']]
    df_mscoco.rename(columns={'id': 'caption_id', 'caption': 'caption_en'}, inplace=True)

    results_data_list = []

    print(f"Scanning {DATA_DIR} for JSON files...")
    json_files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".json"))[:10000]

    for j in tqdm(json_files, desc="Processing JSON files"):
        try:
            with open(os.path.join(DATA_DIR, j), encoding="utf-8") as f:
                meta = json.load(f)

            full_caption = meta["caption_bn"]
            caption_en, caption_bn = extract_captions(full_caption)
            caption_variants = handle_full_stop_variation(caption_en)

            matched_rows = None
            for variant in caption_variants:
                matched_rows = df_mscoco[df_mscoco['caption_en'] == variant]
                if not matched_rows.empty:
                    break

            if matched_rows is None or matched_rows.empty:
                continue

            image_id = matched_rows['image_id'].values[0]
            image_id_str = str(image_id).zfill(12)

            real_image_path = os.path.join(REAL_IMAGE_BASE_DIR, f"COCO_train2014_{image_id_str}.jpg")
            generated_image_path = os.path.join(DATA_DIR, meta["filename"])

            if os.path.exists(real_image_path) and os.path.exists(generated_image_path):
                results_data_list.append({
                    "real_image_path": real_image_path,
                    "generated_image_path": generated_image_path,
                    "caption_bn": caption_bn,
                    "caption_en": caption_en,
                    "valid": True
                })
        except Exception as e:
            print(f"Error processing {j}: {e}")

    df_results = pd.DataFrame(results_data_list)
    print(f"--- Data Preprocessing Complete. Found {len(df_results)} valid pairs. ---")
    if df_results.empty:
        print("No valid data pairs found. Please check your data paths and structure.")
        raise SystemExit
    print(df_results.head())

    # -----------------------------
    # Initialize model and tokenizer
    # -----------------------------
    tokenizer = MBart50TokenizerFast.from_pretrained(
        MBART_MODEL_NAME, src_lang="bn_IN", tgt_lang="bn_IN"
    )
    bn_in_token_id = tokenizer.lang_code_to_id.get("bn_IN")

    # Preflight
    DEBUG_PREFLIGHT = False
    transform_for_preflight = debug_transform if DEBUG_PREFLIGHT else train_transform_safe
    keep_idx = preflight_filter(
        df_results,
        tokenizer,
        image_transform=transform_for_preflight,
        max_length=MAX_CAPTION_LENGTH,
        use_debug=DEBUG_PREFLIGHT
    )
    df_results = df_results.loc[keep_idx].reset_index(drop=True)
    print(f"After preflight: {len(df_results)} usable pairs (from {len(keep_idx)}).")

    # -----------------------------
    # Split last 50 pairs for test
    # -----------------------------
    if len(df_results) > 2000:
        df_train = df_results.iloc[:-50].reset_index(drop=True)
        df_test = df_results.iloc[-50:].reset_index(drop=True)
    else:
        df_train = df_results.copy()
        df_test = pd.DataFrame(columns=df_results.columns)

    dataset = BengaliCaptionDataset(df_train, train_transform_safe, tokenizer, max_length=MAX_CAPTION_LENGTH)
    num_workers = 0
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=True,
        collate_fn=collate_fn,
        num_workers=num_workers,
        persistent_workers=(num_workers > 0)
    )
    
    # Build model with all options
    model = MaxVitMbartCaptioningModel(
        MAXVIT_MODEL_NAME,
        MBART_MODEL_NAME,
        bn_in_token_id=bn_in_token_id,
        patch_alignment_weight=INITIAL_PATCH_ALIGNMENT_WEIGHT,
        attn_temp=PAL_CFG["attn_temp"],
        topk_ratio=PAL_CFG["topk_ratio"],
        use_last_k_cross_layers=PAL_CFG["use_last_k_cross_layers"],
        pool_factor=PAL_CFG["pool_factor"],
        multi_scale=PAL_CFG["multi_scale"],
        use_info_nce=False,
        info_nce_temp=INFO_NCE_CFG["temp"],
        info_nce_coef=INFO_NCE_CFG["coef"],
        use_ot=False,
        ot_reg=OT_CFG["reg"],
        ot_iters=OT_CFG["iters"],
        ot_topk_ratio=OT_CFG["topk_ratio"],
        ot_coef=OT_CFG["coef"]
    ).to(DEVICE)


    for p in model.language_decoder.parameters():
        p.requires_grad = False
    
    # 3) Create optimizer **with ALL params** so later unfreezing gets updated
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    
    # 4) Scheduler
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=LEARNING_RATE,
        epochs=NUM_EPOCHS,               
        steps_per_epoch=len(dataloader),
        pct_start=0.1,
        anneal_strategy='linear',
        cycle_momentum=False
    )
    
    # 5) Grad scaler
    scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))
    
    # 6) Resume if checkpoint exists
    if os.path.exists(MODEL_CHECKPOINT_PATH_IN):
        load_checkpoint_if_any(MODEL_CHECKPOINT_PATH_IN, DEVICE, model, optimizer, scheduler, scaler)
    else:
        print("No MODEL_CHECKPOINT_PATH found; if MODEL_INPUT_PATH exists, load raw weights for warm start.")
        if os.path.exists(MODEL_INPUT_PATH):
            state = torch.load(MODEL_INPUT_PATH, map_location=DEVICE)
            missing, unexpected = model.load_state_dict(state, strict=False)
            print(f"Warm-start from MODEL_INPUT_PATH. Missing: {missing} | Unexpected: {unexpected}")

    RUN_LABEL = "with_pal" if INITIAL_PATCH_ALIGNMENT_WEIGHT > 0.0 else "ce_only"

    history = TrainHistory(
        run_label=RUN_LABEL,
        run_id=RUN_ID,
        meta=dict(
            maxvit=MAXVIT_MODEL_NAME,
            decoder=MBART_MODEL_NAME,
            pal_weight=INITIAL_PATCH_ALIGNMENT_WEIGHT,
            use_nce=USE_INFO_NCE,
            use_ot=USE_OT
        )
    )
    
    # 7) Train exactly 7 epochs on this batch
    train_for_seven_epochs(
        model=model,
        dataloader=dataloader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=DEVICE,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        num_epochs=NUM_EPOCHS,  
        history=history 
    )
    
    save_checkpoint(MODEL_CHECKPOINT_PATH, model, optimizer, scheduler, scaler)

    if HISTORY_CSV_PATH.endswith(".csv"):
        out_hist = HISTORY_CSV_PATH
    else:
        out_hist = f"/kaggle/working/history_{RUN_LABEL}.csv"
    history.to_csv(out_hist)

    try:
        if COMPARE_WITH_CSV and os.path.exists(COMPARE_WITH_CSV):
            # Backward compatible: two single CSVs
            df_this = pd.read_csv(out_hist)
            df_this = df_this.loc[:, ~df_this.columns.astype(str).str.startswith("Unnamed")]
            df_cmp  = pd.read_csv(COMPARE_WITH_CSV)
            df_cmp  = df_cmp.loc[:, ~df_cmp.columns.astype(str).str.startswith("Unnamed")]


            def is_pal(df, fallback_name):
                if "run_label" in df.columns:
                    return any(df["run_label"].astype(str).str.contains("pal", case=False))
                return "pal" in fallback_name.lower()

            if is_pal(df_this, out_hist):
                df_pal_all, df_ce_all = df_this, df_cmp
            elif is_pal(df_cmp, COMPARE_WITH_CSV):
                df_pal_all, df_ce_all = df_cmp, df_this
            else:
                # Fallback heuristic on final loss
                if df_this["total"].iloc[-1] <= df_cmp["total"].iloc[-1]:
                    df_pal_all, df_ce_all = df_this, df_cmp
                else:
                    df_pal_all, df_ce_all = df_cmp, df_this

            plot_pal_vs_ce_dual_from_dfs(
                df_pal_all=df_pal_all, df_ce_all=df_ce_all,
                out_pdf=LOSS_FIG_PATH, logy_total=LOGY_TOTAL, inset_epochs=INSET_EPOCHS
            )
        else:
            print("Provide PAL_HIST_GLOB & CE_HIST_GLOB for CI bands, or set COMPARE_WITH_CSV for a single comparison.")
    
    except Exception as e:
        print(f"Could not generate comparison figure automatically: {e}")


    # -----------------------------
    # Embedding visualization (CE-only vs PAL)
    # -----------------------------
    try:
        # Prefer a held-out set
        if not df_test.empty:
            eval_dataset = BengaliCaptionDataset(df_test, train_transform_safe, tokenizer, max_length=MAX_CAPTION_LENGTH)
        else:
            # fallback: a small slice from train
            eval_dataset = BengaliCaptionDataset(df_train.sample(min(200, len(df_train), random_state=SEED) if len(df_train)>0 else df_train),
                                                 train_transform_safe, tokenizer, max_length=MAX_CAPTION_LENGTH)
    
        # g_eval = torch.Generator(); g_eval.manual_seed(SEED)
        eval_loader = DataLoader(
            eval_dataset,
            batch_size=64,              # try 64 or 128 if VRAM allows
            shuffle=False,              # deterministic ordering for reproducibility
            pin_memory=True,
            collate_fn=collate_fn,
            num_workers=2
        )
    
        if CE_ONLY_CKPT_IN and WITH_PAL_CKPT_IN:
            visualize_embeddings_ce_vs_pal(
                dataloader_eval=eval_loader,
                ce_ckpt_path=CE_ONLY_CKPT_IN,
                pal_ckpt_path=WITH_PAL_CKPT_IN,
                method=EMBED_METHOD,                 # "umap" or "tsne"
                pdf_path=EMBED_PDF_PATH,             # where to save
                max_batches=EMBED_MAX_BATCHES,
                random_state=EMBED_RANDOM_STATE
            )
        else:
            print("Set CE_ONLY_CKPT_IN and WITH_PAL_CKPT_IN to generate CE vs PAL embedding plots.")
    except Exception as e:
        print(f"Embedding visualization skipped: {e}")

    
    
    print("Incremental run complete. Ready for the next 15–20k batch.")
